In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python 버전:", sys.version.split()[0])

packages = [
    "torch",
    "torchaudio",
    "transformers",
    "accelerate",
    "huggingface-hub",
]

for package in packages:
    try:
        print(f"{package}, {version(package)}")
    except PackageNotFoundError:
        print(f"{package} 설치되지 않음.")

In [ ]:
!nvidia-smi

In [ ]:
%pip install torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch

print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
    print("GPU 계산 확인:", (torch.ones(1, device="cuda") + 1).item())

In [ ]:
%pip install numpy==1.26.4 transformers==4.53.3 accelerate==1.8.1 librosa==0.11.0 soundfile==0.13.1

In [ ]:
import torch
import numpy as np
import soundfile as sf

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

print("GPU 사용 가능:", torch.cuda.is_available())
print("Whisper 준비 모듈 불러오기 성공!")

In [ ]:
from pathlib import Path
import os
import subprocess

# chap05 폴더 위치 찾기
current_dir = Path.cwd()

if current_dir.name == "chap05":
    chap05_dir = current_dir
elif (current_dir / "chap05").is_dir():
    chap05_dir = current_dir / "chap05"
else:
    raise FileNotFoundError(f"chap05를 찾지 못했습니다: {current_dir}")

# FFmpeg 위치 설정
ffmpeg_bin = chap05_dir / "ffmpeg" / "bin"
ffmpeg_exe = ffmpeg_bin / "ffmpeg.exe"

if not ffmpeg_exe.is_file():
    raise FileNotFoundError(f"이 위치에 파일이 없습니다: {ffmpeg_exe}")

os.environ["PATH"] = str(ffmpeg_bin) + os.pathsep + os.environ["PATH"]

# 실행 확인
check = subprocess.run(
    [str(ffmpeg_exe), "-version"],
    capture_output=True,
    text=True,
    errors="replace",
    check=True,
)

print(check.stdout.splitlines()[0])
print("FFmpeg 설정 완료!")

In [ ]:
audio_path = chap05_dir / "audio" / "lsy_audio_2023_58s.mp3"

if not audio_path.is_file():
    raise FileNotFoundError(f"음성 파일이 없습니다: {audio_path}")

print("음성을 텍스트로 변환하는 중...")

result = transcriber(
    str(audio_path),
    return_timestamps=True,
    generate_kwargs={
        "language": "korean",
        "task": "transcribe",
    },
)

print(result["text"])
print("\n시간 구간 개수:", len(result["chunks"]))

In [ ]:
import csv

# chap05 안에 output 폴더 만들기
output_dir = chap05_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "lsy_audio_2023_58s.csv"

# 시간 구간별 결과 저장
with csv_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file, delimiter="|")
    writer.writerow(["start", "end", "text"])

    for chunk in result["chunks"]:
        start, end = chunk["timestamp"]
        writer.writerow([start, end, chunk["text"].strip()])

print("CSV 파일 저장 완료")
print("저장 위치:", csv_path)
print("저장된 구간 수:", len(result["chunks"]))

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

model_id = "openai/whisper-large-v3-turbo"

print("모델 다운로드 및 불러오기 시작...")

# GPU 메모리 사용을 줄이기 위한 float16 사용
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)

model = model.to("cuda")
processor = AutoProcessor.from_pretrained(model_id)

# 음성 인식 기능 준비
transcriber = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch.float16,
    device=0,
    batch_size=1,
)

print("Whisper 모델 준비 완료.")

In [ ]:
from pathlib import Path
import soundfile as sf
import torch
import torchaudio
import textwrap

# 작업 위치가 chap05 또는 프로젝트 최상위인 경우 처리
audio_path = Path("audio/lsy_audio_2023_58s.mp3")

if not audio_path.is_file():
    audio_path = Path("chap05/audio/lsy_audio_2023_58s.mp3")

if not audio_path.is_file():
    raise FileNotFoundError(
        f"MP3 파일을 찾지 못했습니다. 현재 작업 폴더: {Path.cwd()}"
    )

# MP3를 숫자로 이루어진 음성 데이터로 읽기
audio, sample_rate = sf.read(
    str(audio_path),
    dtype="float32",
    always_2d=True,
)

# 스테레오 등 여러 채널을 모노로 합치기
audio = audio.mean(axis=1)

# Whisper가 사용하는 샘플링 주파수로 변환
target_rate = 16000

if sample_rate != target_rate:
    audio = torchaudio.functional.resample(
        torch.from_numpy(audio),
        orig_freq=sample_rate,
        new_freq=target_rate,
    ).numpy()

print("음성을 텍스트로 변환하는 중...")

# 30초가 넘는 음성도 처리하도록 시간 정보 활성화
result = transcriber(
    {"raw": audio, "sampling_rate": target_rate},
    return_timestamps=True,
    generate_kwargs={
        "language": "korean",
        "task": "transcribe",
    },
)

local_text = result["text"]

print("\n변환 결과:\n")
print(textwrap.fill(local_text, width=60))

In [ ]:
import sys
import numpy as np
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

print("실행 환경:", sys.executable)
print("NumPy 버전:", np.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

# NumPy와 PyTorch 연결 확인
data = torch.from_numpy(np.array([1.0, 2.0], dtype=np.float32))

if torch.cuda.is_available():
    print("GPU 계산 결과:", (data.to("cuda") + 1).cpu().numpy())

print("기존 환경 확인 완료!")

In [ ]:
import os
import csv
import subprocess
from pathlib import Path

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


def main():
    # 1. chap05 폴더 찾기
    current_dir = Path.cwd()

    if current_dir.name == "chap05":
        chap05_dir = current_dir
    elif (current_dir / "chap05").is_dir():
        chap05_dir = current_dir / "chap05"
    else:
        raise FileNotFoundError(f"chap05 폴더를 찾지 못했습니다: {current_dir}")

    audio_path = chap05_dir / "audio" / "lsy_audio_2023_58s.mp3"
    ffmpeg_bin = chap05_dir / "ffmpeg" / "bin"
    ffmpeg_exe = ffmpeg_bin / "ffmpeg.exe"

    if not audio_path.is_file():
        raise FileNotFoundError(f"음성 파일이 없습니다: {audio_path}")

    if not ffmpeg_exe.is_file():
        raise FileNotFoundError(f"FFmpeg 파일이 없습니다: {ffmpeg_exe}")

    # 2. FFmpeg 설정
    os.environ["PATH"] = str(ffmpeg_bin) + os.pathsep + os.environ["PATH"]

    subprocess.run(
        [str(ffmpeg_exe), "-version"],
        capture_output=True,
        check=True,
    )
    print("① FFmpeg 준비 완료")

    # 3. GPU 확인 및 모델 준비
    if not torch.cuda.is_available():
        raise RuntimeError("GPU를 사용할 수 없습니다. 실행 환경을 확인해 주세요.")

    model_id = "openai/whisper-large-v3-turbo"

    print("② Whisper 모델을 불러오는 중...")

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    ).to("cuda")

    processor = AutoProcessor.from_pretrained(model_id)

    transcriber = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch.float16,
        device=0,
        batch_size=1,
    )

    # 4. 음성을 텍스트로 변환
    print("③ 음성을 텍스트로 변환하는 중...")

    result = transcriber(
        str(audio_path),
        return_timestamps=True,
        generate_kwargs={
            "language": "korean",
            "task": "transcribe",
        },
    )

    chunks = result.get("chunks", [])
    if not chunks:
        raise RuntimeError("시간 구간이 없어 CSV를 저장하지 않았습니다.")

    print("\n[변환 결과]")
    print(result["text"])

    # 5. output 폴더 생성 및 CSV 저장
    output_dir = chap05_dir / "output"
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "lsy_audio_2023_58.csv"

    with csv_path.open("w", newline="", encoding="utf-8-sig") as file:
        writer = csv.writer(file, delimiter="|")
        writer.writerow(["start", "end", "text"])

        for chunk in chunks:
            start, end = chunk["timestamp"]
            writer.writerow([start, end, chunk["text"].strip()])

    print("\n④ CSV 저장 완료!")
    print("저장 위치:", csv_path)
    print("저장된 구간 수:", len(chunks))


# 위 과정을 한 번에 실행
main()